# Partnership Letters 1957–1970

29 letters Buffett wrote to his limited partners. Pages 0–154 of the big PDF only — everything after that is Berkshire shareholder era which we're covering via Cunningham instead.

In [1]:
import sys, re, statistics, importlib
from pathlib import Path
from collections import Counter

sys.path.insert(0, "..")
import pipeline.core
importlib.reload(pipeline.core)
from pipeline.core import (
    Chunk, QAPair, LABELS, extract_text,
    classify_chunks, generate_all, score_all,
    filter_by_quality, coverage_audit,
    export_csv, export_detailed,
    save_checkpoint, load_checkpoint,
)
import fitz

## Chunking

Two header formats to detect: `XXXX Letter` for the early years (1957–1960) and `BUFFETT PARTNERSHIP, LTD.` for 1961–1970. Paragraphs that are mostly numbers get filtered — those are performance tables, not Buffett's words. Validated in `test_partnership_chunking.py`.

In [2]:
def extract_partnership_text(pdf_path):
    doc = fitz.open(pdf_path)
    pages = [doc[pg].get_text() for pg in range(min(155, len(doc)))]
    doc.close()
    return "\n".join(pages)


def find_letter_boundaries(text):
    boundaries = []

    for m in re.finditer(r'^(\d{4})\s+Letter', text, re.MULTILINE):
        boundaries.append({"pos": m.start(), "year": m.group(1), "date": None, "pattern": "year_letter"})

    date_re = re.compile(
        r'((?:January|February|March|April|May|June|July|August|September|October|November|December)'
        r'\s*\d{0,2}(?:st|nd|rd|th)?\s*,?\s*\d{4})'
    )
    for m in re.finditer(r'BUFFETT PARTNERSHIP[.,] LTD\.', text):
        context = text[m.start():m.start() + 300]
        date_match = date_re.search(context)
        if date_match:
            date_str = date_match.group(1)
        else:
            month_year = re.search(
                r'((?:January|February|March|April|May|June|July|August|September|October|November|December),?\s*\d{4})',
                context
            )
            date_str = month_year.group(1) if month_year else "UNKNOWN"
        year = re.search(r'(\d{4})', date_str).group(1) if re.search(r'(\d{4})', date_str) else "UNKNOWN"
        boundaries.append({"pos": m.start(), "year": year, "date": date_str, "pattern": "buffett_partnership"})

    boundaries.sort(key=lambda b: b["pos"])
    deduped = []
    for b in boundaries:
        if deduped and abs(b["pos"] - deduped[-1]["pos"]) < 500:
            if b["date"] and not deduped[-1]["date"]:
                deduped[-1] = b
        else:
            deduped.append(b)
    return deduped


def split_into_letters(text, boundaries):
    letters = []
    for i, b in enumerate(boundaries):
        start = b["pos"]
        end = boundaries[i + 1]["pos"] if i + 1 < len(boundaries) else len(text)
        letter_text = text[start:end].strip()
        letters.append({"text": letter_text, "year": b["year"], "date": b.get("date", "")})
    return letters


def is_financial_table(text, threshold=0.20):
    if len(text) < 50:
        return True
    noise_chars = sum(1 for c in text if c.isdigit() or c in '$%')
    return (noise_chars / len(text)) > threshold


def _sub_chunk(text, source_file, section, max_chars=4000, overlap=500):
    if len(text) <= max_chars:
        return [Chunk(text=text, source_file=source_file,
                      source_section=section, chunk_strategy="chronological")]
    paragraphs = [p.strip() for p in text.split('\n') if len(p.strip()) > 30]
    sub_chunks, current, current_len = [], [], 0
    for para in paragraphs:
        if current_len + len(para) > max_chars and current:
            sub_chunks.append(Chunk(
                text='\n'.join(current), source_file=source_file,
                source_section=section, chunk_strategy="chronological_subchunk"))
            overlap_paras, overlap_len = [], 0
            for p in reversed(current):
                if overlap_len + len(p) > overlap: break
                overlap_paras.insert(0, p)
                overlap_len += len(p)
            current = overlap_paras + [para]
            current_len = overlap_len + len(para)
        else:
            current.append(para)
            current_len += len(para)
    if current:
        sub_chunks.append(Chunk(
            text='\n'.join(current), source_file=source_file,
            source_section=section,
            chunk_strategy="chronological_subchunk" if sub_chunks else "chronological"))
    return sub_chunks


def chunk_partnership_letters(pdf_path):
    raw = extract_partnership_text(pdf_path)
    raw = re.sub(r'^\d{1,4}\s*$', '', raw, flags=re.MULTILINE)

    boundaries = find_letter_boundaries(raw)
    letters = split_into_letters(raw, boundaries)

    all_chunks = []
    skipped_tables = 0

    for letter in letters:
        section_name = f"{letter['year']} Letter"
        if letter["date"]:
            section_name += f" ({letter['date']})"

        paragraphs = [p.strip() for p in letter["text"].split('\n') if len(p.strip()) > 30]
        clean_paragraphs = []
        for para in paragraphs:
            if is_financial_table(para):
                skipped_tables += 1
            else:
                clean_paragraphs.append(para)

        if not clean_paragraphs:
            continue

        clean_text = '\n'.join(clean_paragraphs)
        all_chunks.extend(_sub_chunk(clean_text, "partnership_letters.pdf", section_name))

    merged = []
    for chunk in all_chunks:
        if len(chunk.text) < 1000 and merged:
            merged[-1].text += '\n' + chunk.text
        else:
            merged.append(chunk)

    print(f"Letters found: {len(letters)} | Tables filtered: {skipped_tables} | Final chunks: {len(merged)}")
    return merged

In [3]:
chunks = chunk_partnership_letters(Path("../sources/Warren-Buffett-Berkshire-Letters-1957-2012.pdf"))

Letters found: 29 | Tables filtered: 239 | Final chunks: 122


## Classification
Each chunk gets assigned one of the 6 labels via DeepSeek. Pre-labeled chunks (if any) skip the LLM call. Progress prints every 10 chunks.

In [5]:
classified = await classify_chunks(chunks)
save_checkpoint(classified, "letters_classified")

Classifying 122 chunks (skipping 0 pre-labeled)
  10/122
  20/122
  30/122
  40/122
  50/122
  60/122
  70/122
  80/122
  90/122
  100/122
  110/122
  120/122
  122/122

Label distribution:
  Strategy Development: 89
  Risk Management: 18
  Psychology: 7
  Timing: 4
  Personal Life: 3
  Adaptability: 1
Saved: C:\Users\Admin\OneDrive\Desktop\buffet-qa-pipeline\intermediate\letters_classified.pkl


In [6]:
dist = Counter(c.label for c in classified if c.label)
print("Label distribution:\n")
for label, count in dist.most_common():
    bar = "█" * count
    print(f"  {label:25s} {count:3d} {bar}")

failed = [c for c in classified if c.label is None]
if failed:
    print(f"\n!! {len(failed)} chunks unclassified")

Label distribution:

  Strategy Development       89 █████████████████████████████████████████████████████████████████████████████████████████
  Risk Management            18 ██████████████████
  Psychology                  7 ███████
  Timing                      4 ████
  Personal Life               3 ███
  Adaptability                1 █


## Generation
Each classified chunk gets sent to DeepSeek with a label-specific prompt that enforces grounded answers, 3–5 sentence self-contained responses, and sublabel coverage. 3 candidate pairs per chunk. Checkpoints after completion so scoring can be re-run independently if needed.

In [7]:
pairs = await generate_all(classified, n_pairs=3)
save_checkpoint(pairs, "letters_pairs_raw")
print(f"\nTotal raw pairs: {len(pairs)}")

  5/122 chunks | 15 pairs
  10/122 chunks | 30 pairs
  15/122 chunks | 45 pairs
  20/122 chunks | 60 pairs
  25/122 chunks | 75 pairs
  30/122 chunks | 90 pairs
  35/122 chunks | 105 pairs
  40/122 chunks | 120 pairs
  45/122 chunks | 135 pairs
  50/122 chunks | 150 pairs
  55/122 chunks | 165 pairs
  60/122 chunks | 180 pairs
  65/122 chunks | 195 pairs
  70/122 chunks | 210 pairs
  75/122 chunks | 225 pairs
  80/122 chunks | 240 pairs
  85/122 chunks | 255 pairs
  90/122 chunks | 270 pairs
  95/122 chunks | 285 pairs
  100/122 chunks | 300 pairs
  105/122 chunks | 315 pairs
  110/122 chunks | 330 pairs
  115/122 chunks | 345 pairs
  120/122 chunks | 360 pairs
  122/122 chunks | 366 pairs
Saved: C:\Users\Admin\OneDrive\Desktop\buffet-qa-pipeline\intermediate\letters_pairs_raw.pkl

Total raw pairs: 366


In [8]:
seen_labels = set()
for p in pairs:
    if p.label in seen_labels:
        continue
    seen_labels.add(p.label)
    print(f"── {p.label} ──")
    print(f"  Q: {p.question}")
    print(f"  A: {p.answer[:250]}...")
    print()

── Timing ──
  Q: According to Buffett in this passage, what are the two main categories of investments he makes, and how does their proportion change based on market conditions?
  A: Buffett categorizes investments into "general issues" (substantially undervalued securities) and "work-outs" (investments dependent on specific corporate actions like sales or mergers). He adjusts the portfolio mix based on market valuation. After a ...

── Strategy Development ──
  Q: In his 1957 letter, how does Warren Buffett explain the performance differences between his three 1956 partnerships, and what principle does he highlight?
  A: Buffett explains that the third partnership, started latest in 1956, performed vastly better (up 25%) than the first two (up 6.2% and 7.8%) because it began when the market was lower and several securities were particularly attractive. Since funds be...

── Psychology ──
  Q: How does Warren Buffett describe the general market psychology in 1958, and what is his conc

## Scoring & filtering
Each raw pair gets scored by DeepSeek against its source chunk on groundedness, label fit, richness, and novelty. Composite score is a weighted average (groundedness 0.35, label fit 0.25, richness 0.20, novelty 0.20). Pairs below 0.7 get dropped. The chunk map links each pair back to the text it was generated from so the scorer can verify claims.

In [9]:
chunk_map = {c.chunk_id: c for c in classified}
scored = await score_all(pairs, chunk_map)
filtered = filter_by_quality(scored, threshold=0.7)
save_checkpoint(filtered, "letters_pairs_filtered")

  Scored 10/366
  Scored 20/366
  [WARN] Scoring failed: Extra data: line 3 column 1 (char 74)
  Scored 30/366
  Scored 40/366
  Scored 50/366
  Scored 60/366
  Scored 70/366
  Scored 80/366
  Scored 90/366
  Scored 100/366
  Scored 110/366
  Scored 120/366
  Scored 130/366
  Scored 140/366
  Scored 150/366
  Scored 160/366
  Scored 170/366
  Scored 180/366
  Scored 190/366
  Scored 200/366
  Scored 210/366
  Scored 220/366
  Scored 230/366
  Scored 240/366
  Scored 250/366
  Scored 260/366
  Scored 270/366
  Scored 280/366
  Scored 290/366
  Scored 300/366
  Scored 310/366
  Scored 320/366
  Scored 330/366
  Scored 340/366
  Scored 350/366
  Scored 360/366
  Scored 366/366
Quality filter: 265/366 passed (threshold=0.7)
Saved: C:\Users\Admin\OneDrive\Desktop\buffet-qa-pipeline\intermediate\letters_pairs_filtered.pkl


In [10]:
scores = [p.composite_score for p in filtered if p.composite_score]
print(f"Pairs after filtering: {len(filtered)} / {len(pairs)} raw")
print(f"Drop rate: {(1 - len(filtered)/len(pairs))*100:.1f}%\n")
print(f"Score range: {min(scores):.2f} — {max(scores):.2f}")
print(f"Mean:   {statistics.mean(scores):.2f}")
print(f"Median: {statistics.median(scores):.2f}")

Pairs after filtering: 265 / 366 raw
Drop rate: 27.6%

Score range: 0.71 — 0.91
Mean:   0.83
Median: 0.84


## Coverage & export
Reports what this document contributed — pair count per label, sublabel coverage, and percentage breakdown. No gap analysis here; that's the assembly notebook's job after all 5 sources are merged. Exports both the clean assignment CSV and a detailed version with scores and metadata for debugging.

In [14]:
print("Partnership letters contribution:\n")
report = coverage_audit(filtered)

Partnership letters contribution:

  Personal Life               7 pairs ( 2.6%)
    Sublabels hit: habits, family_influence, personal_values
  Strategy Development      190 pairs (71.7%)
    Sublabels hit: value_investing_framework, margin_of_safety, competitive_moat, business_quality, circle_of_competence, capital_allocation
  Timing                      9 pairs ( 3.4%)
    Sublabels hit: entry_criteria, exit_criteria, market_valuation, opportunity_cost, patience, price_vs_value
  Risk Management            38 pairs (14.3%)
    Sublabels hit: position_sizing, diversification, leverage_avoidance, permanent_loss, margin_of_safety_risk, concentration
  Adaptability                3 pairs ( 1.1%)
    Sublabels hit: crisis_response, strategy_evolution, new_opportunities
  Psychology                 18 pairs ( 6.8%)
    Sublabels hit: temperament, emotional_discipline, contrarian_thinking, fear_greed, independence, rationality

  Total: 265 pairs


In [15]:
export_csv(filtered, Path("../intermediate/letters_qa.csv"))
export_detailed(filtered, Path("../intermediate/letters_qa_detailed.csv"))

Exported 265 pairs to ..\intermediate\letters_qa.csv
  Strategy Development: 190
  Risk Management: 38
  Psychology: 18
  Timing: 9
  Personal Life: 7
  Adaptability: 3
Detailed export: ..\intermediate\letters_qa_detailed.csv


In [16]:
label_counts = Counter(p.label for p in filtered)
print(f"FINAL: {len(filtered)} pairs across {len(label_counts)} labels\n")
for label, count in label_counts.most_common():
    best = max((p for p in filtered if p.label == label), key=lambda p: p.composite_score)
    print(f"── {label} ({count} pairs, best: {best.composite_score:.2f}) ──")
    print(f"  Q: {best.question}")
    print(f"  A: {best.answer[:300]}")
    print()

FINAL: 265 pairs across 6 labels

── Strategy Development (190 pairs, best: 0.91) ──
  Q: In his 1966 letter, what does Warren Buffett's experience with a 'Generals - Relatively Undervalued' stock reveal about his strategy for building a position?
  A: Buffett's strategy emphasized patience and the accumulation of a substantial position over time. In 1966, he found one important new idea in this category and began buying in late spring. However, the stock price rose relatively quickly due to outside conditions, limiting his accumulation to about $

── Risk Management (38 pairs, best: 0.91) ──
  Q: In the 1962 letter, what specific target does Buffett set for his partnership's performance during a market decline, and what does achieving this target signify about its risk profile?
  A: Buffett states the partnership's target is "an approximately 1/2% decline for each 1% decline in the Dow." He explains that achieving this target means they have "a considerably more conservative vehicle f

In [17]:
# classified = load_checkpoint("letters_classified")
# pairs = load_checkpoint("letters_pairs_raw")
# filtered = load_checkpoint("letters_pairs_filtered")